# Step 7.6 — Final Model Selection

**Financial Fraud Detection System**

---

## Objective

This notebook documents the **final model-selection decision** for the Financial Fraud Detection project based on the actual test-set results observed in the completed baseline experiments (Steps 4–7).

**Scope:**
- Verify actual metrics from the three baseline model notebooks.
- Apply fraud-detection-appropriate selection criteria (prioritizing Recall and F1 over Accuracy).
- Select the strongest available baseline candidate.
- Clearly distinguish between "best available baseline" and "production-ready model".
- Document all limitations, including data leakage concerns and class-imbalance handling.

**Not in scope (Step 7.7+):**
- No model retraining, tuning, or serialization.
- No SMOTE, threshold optimization, or new experiments.
- No dashboard, deployment, or model artifact creation.

---

## 1. Selection Scope

| Item | Detail |
|---|---|
| Dataset | `synthetic_fraud_dataset1 (1).csv` |
| Rows | 50,000 |
| Target | `Fraud_Label` |
| Class distribution | ~67.87% non-fraud / ~32.13% fraud |
| Train/test split | 80/20, `random_state=42`, `stratify=y` |
| Resampling | None (no SMOTE applied in baselines) |
| Models evaluated | Logistic Regression, Random Forest, XGBoost |
| Selection basis | **Test set performance only** |

---

## 2. Source Notebook Verification

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

# Verify all source notebooks exist
notebook_dir = os.path.join(os.path.dirname(os.getcwd()), 'notebooks')
# Handle case where we are already in the notebooks directory
if not os.path.isdir(notebook_dir):
    notebook_dir = os.getcwd()
# Also handle case where CWD is the project root
if not os.path.isfile(os.path.join(notebook_dir, '04_logistic_regression_baseline.ipynb')):
    alt_dir = os.path.join(os.getcwd(), 'notebooks')
    if os.path.isdir(alt_dir):
        notebook_dir = alt_dir

required_notebooks = [
    '04_logistic_regression_baseline.ipynb',
    '05_random_forest_model.ipynb',
    '06_xgboost_model.ipynb',
    '07_model_evaluation_comparison.ipynb'
]

print('Source Notebook Verification')
print('=' * 60)
all_exist = True
for nb in required_notebooks:
    path = os.path.join(notebook_dir, nb)
    exists = os.path.isfile(path)
    status = 'FOUND' if exists else 'MISSING'
    print(f'  {nb}: {status}')
    if not exists:
        all_exist = False

print()
if all_exist:
    print('All source notebooks verified successfully.')
else:
    print('WARNING: One or more source notebooks are missing!')

---

## 3. Verified Model Results

The following metrics were extracted directly from the individual model notebooks and confirmed against the Step 7.5 comparison notebook (`07_model_evaluation_comparison.ipynb`).

**No values were fabricated. All values below are exact reproductions of notebook outputs.**

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# Verified test-set metrics from source notebooks
# Sources: 04, 05, 06, 07 notebooks
# ============================================================

test_results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Test Accuracy':  [0.6787, 0.6775, 0.6725],
    'Test Precision': [0.0000, 0.3333, 0.3315],
    'Test Recall':    [0.0000, 0.0037, 0.0190],
    'Test F1':        [0.0000, 0.0074, 0.0359],
    'Test ROC-AUC':   [0.4903, 0.5023, 0.4944],
    'Test PR-AUC':    [np.nan,  0.3275, 0.3171]
})

# ============================================================
# Verified training-set metrics from source notebooks
# Sources: 04 (LR), 05 (RF), 06 (XGB) notebooks
# ============================================================

train_results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Train Accuracy':  [0.6786, 1.0000, 0.7144],
    'Train Precision': [0.0000, 1.0000, 0.9857],
    'Train Recall':    [0.0000, 1.0000, 0.1127],
    'Train F1':        [0.0000, 1.0000, 0.2023],
    'Train ROC-AUC':   [0.5138, 1.0000, 0.8570]
})

print('Verified Test-Set Performance')
print('=' * 90)
print(test_results.to_string(index=False))

print()
print('Verified Training-Set Performance')
print('=' * 90)
print(train_results.to_string(index=False))

---

## 4. Model Selection Criteria

For a **fraud detection** system, the selection priorities are:

| Priority | Metric | Rationale |
|---|---|---|
| **PRIMARY** | Test Recall | Missing a fraudulent transaction (false negative) is the costliest error |
| **PRIMARY** | Test F1 | Balances precision and recall for the minority fraud class |
| Secondary | Test Precision | High precision reduces false alarms |
| Secondary | Test PR-AUC | Summarizes precision-recall trade-off across thresholds |
| Secondary | Test ROC-AUC | Measures overall discrimination ability |
| Context | Test Accuracy | Misleading in imbalanced datasets; used only as context |

**Additional considerations:**
- Train/test generalization gap (evidence of overfitting)
- Practical ability to detect fraudulent transactions
- Data quality and leakage concerns

**Decision approach:** Transparent qualitative assessment supported by actual metrics. No artificial numerical scoring formula is imposed.

---

## 5. Test Performance Analysis

In [ ]:
print('Test Performance Ranking by Primary Criteria')
print('=' * 60)

# Rank by Test Recall
recall_ranked = test_results[['Model', 'Test Recall']].sort_values('Test Recall', ascending=False)
print('\nRanked by Test Recall (PRIMARY):')
for i, (_, row) in enumerate(recall_ranked.iterrows(), 1):
    print(f'  {i}. {row["Model"]}: {row["Test Recall"]:.4f}')

# Rank by Test F1
f1_ranked = test_results[['Model', 'Test F1']].sort_values('Test F1', ascending=False)
print('\nRanked by Test F1 (PRIMARY):')
for i, (_, row) in enumerate(f1_ranked.iterrows(), 1):
    print(f'  {i}. {row["Model"]}: {row["Test F1"]:.4f}')

# Rank by Test Precision
prec_ranked = test_results[['Model', 'Test Precision']].sort_values('Test Precision', ascending=False)
print('\nRanked by Test Precision (Secondary):')
for i, (_, row) in enumerate(prec_ranked.iterrows(), 1):
    print(f'  {i}. {row["Model"]}: {row["Test Precision"]:.4f}')

# Rank by Test ROC-AUC
auc_ranked = test_results[['Model', 'Test ROC-AUC']].sort_values('Test ROC-AUC', ascending=False)
print('\nRanked by Test ROC-AUC (Secondary):')
for i, (_, row) in enumerate(auc_ranked.iterrows(), 1):
    print(f'  {i}. {row["Model"]}: {row["Test ROC-AUC"]:.4f}')

# Rank by Test PR-AUC (excluding NaN)
prauc_ranked = test_results[['Model', 'Test PR-AUC']].dropna().sort_values('Test PR-AUC', ascending=False)
print('\nRanked by Test PR-AUC (Secondary):')
for i, (_, row) in enumerate(prauc_ranked.iterrows(), 1):
    print(f'  {i}. {row["Model"]}: {row["Test PR-AUC"]:.4f}')
print('  Note: Logistic Regression PR-AUC was not computed in its source notebook.')

print()
print('Key Observation:')
print('  All three models exhibit very poor fraud detection on the test set.')
print('  The highest test Recall is only 0.0190 (XGBoost), meaning ~98% of')
print('  fraudulent transactions are missed.')

---

## 6. Generalization / Overfitting Analysis

In [ ]:
# Compute train-test gaps using verified source notebook values
gap_data = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'F1 Gap (Train-Test)': [
        0.0000 - 0.0000,  # LR: Train F1=0.0000, Test F1=0.0000
        1.0000 - 0.0074,  # RF: Train F1=1.0000, Test F1=0.0074
        0.2023 - 0.0359   # XGB: Train F1=0.2023, Test F1=0.0359
    ],
    'ROC-AUC Gap (Train-Test)': [
        0.5138 - 0.4903,  # LR: Train=0.5138, Test=0.4903
        1.0000 - 0.5023,  # RF: Train=1.0000, Test=0.5023
        0.8570 - 0.4944   # XGB: Train=0.8570, Test=0.4944
    ],
    'Recall Gap (Train-Test)': [
        0.0000 - 0.0000,  # LR: Train=0.0000, Test=0.0000
        1.0000 - 0.0037,  # RF: Train=1.0000, Test=0.0037
        0.1127 - 0.0190   # XGB: Train=0.1127, Test=0.0190
    ]
})

print('Train-Test Generalization Gaps')
print('=' * 80)
print(gap_data.to_string(index=False))

print()
print('Interpretation:')
print('-' * 80)
print('Logistic Regression:')
print('  - Minimal train-test gaps (F1 gap: 0.0000, ROC-AUC gap: +0.0235)')
print('  - NOT overfitting — the model is UNDERFITTING')
print('  - Behaves as a majority-class classifier on both train and test sets')
print()
print('Random Forest:')
print('  - SEVERE overfitting: perfect training scores (all 1.0000)')
print('  - F1 gap: 0.9926, ROC-AUC gap: 0.4977, Recall gap: 0.9963')
print('  - Memorized training data completely; fails to generalize')
print()
print('XGBoost:')
print('  - Significant overfitting, but less extreme than Random Forest')
print('  - F1 gap: 0.1664, ROC-AUC gap: 0.3626, Recall gap: 0.0937')
print('  - Shows some learning signal on training data but poor generalization')
print('  - Smallest relative overfitting gap among models that learned any signal')

---

## 7. Model-by-Model Assessment

### 7.1 Logistic Regression

**Why NOT selected:**

| Criterion | Assessment |
|---|---|
| Test Recall | **0.0000** — detects zero fraudulent transactions |
| Test F1 | **0.0000** — no meaningful fraud detection |
| Test Precision | 0.0000 — undefined (no positive predictions) |
| Test ROC-AUC | 0.4903 — below random chance (0.50) |
| Generalization | Not overfitting, but severely underfitting |
| Behavior | Acts as a majority-class (non-fraud) classifier |

**Conclusion:** Logistic Regression is functionally useless for fraud detection in this experiment. It predicts every transaction as non-fraud. While it achieves the highest accuracy (0.6787), this is entirely due to the non-fraud majority class.

---

### 7.2 Random Forest

**Why NOT selected:**

| Criterion | Assessment |
|---|---|
| Test Recall | **0.0037** — detects only ~0.37% of fraud |
| Test F1 | **0.0074** — negligible fraud detection |
| Test Precision | 0.3333 — highest among models, but based on only 12 correct predictions out of 3,213 fraud cases |
| Test ROC-AUC | 0.5023 — highest, but barely above random chance |
| Test PR-AUC | 0.3275 — highest, but still poor |
| Generalization | **SEVERE overfitting** — all training metrics = 1.0000 |

**Conclusion:** Random Forest shows the most extreme overfitting: perfect training performance coupled with near-zero test fraud detection. Despite marginally higher ROC-AUC and PR-AUC, its recall is practically zero. The model memorized training data but cannot generalize.

---

### 7.3 XGBoost

**Why selected as the strongest available baseline:**

| Criterion | Assessment |
|---|---|
| Test Recall | **0.0190** — highest among all models (~5× higher than Random Forest) |
| Test F1 | **0.0359** — highest among all models (~5× higher than Random Forest) |
| Test Precision | 0.3315 — comparable to Random Forest |
| Test ROC-AUC | 0.4944 — slightly below Random Forest, below 0.50 |
| Test PR-AUC | 0.3171 — slightly below Random Forest |
| Generalization | Significant overfitting (ROC-AUC gap: 0.3626), but less extreme than RF |

**Conclusion:** XGBoost is the only model that shows any meaningful (albeit weak) fraud detection on the test set. It ranks first on both primary criteria (Recall and F1) by a substantial relative margin. While its ROC-AUC is below 0.50 and PR-AUC is slightly lower than Random Forest, these secondary metrics are outweighed by the primary fraud-detection criteria.

---

## 8. Model Selection Decision Table

In [ ]:
# Final decision table
decision_table = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Test Recall':    [0.0000, 0.0037, 0.0190],
    'Test F1':        [0.0000, 0.0074, 0.0359],
    'Test Precision': [0.0000, 0.3333, 0.3315],
    'Test ROC-AUC':   [0.4903, 0.5023, 0.4944],
    'Test PR-AUC':    ['N/A',  0.3275, 0.3171],
    'Generalization': ['Underfitting', 'Severe Overfitting', 'Significant Overfitting'],
    'Decision':       ['Not Selected', 'Not Selected', 'Selected Candidate']
})

print('Final Model Selection Decision Table')
print('=' * 120)
print(decision_table.to_string(index=False))

---

## 9. Final Model Selection Decision

### Selected Model: XGBoost

### Reason for Selection

XGBoost is selected as the **final candidate model** for the next project stage because it achieved the **highest observed test Recall (0.0190) and test F1 (0.0359)** among the three evaluated baseline models. In a fraud detection context, Recall and F1 are the primary selection criteria because missing fraudulent transactions (false negatives) carries the highest cost.

### Key Test Metrics

| Metric | Value |
|---|---|
| Test Recall | 0.0190 |
| Test F1 | 0.0359 |
| Test Precision | 0.3315 |
| Test ROC-AUC | 0.4944 |
| Test PR-AUC | 0.3171 |
| Test Accuracy | 0.6725 |

### Generalization Assessment

XGBoost exhibits **significant overfitting**: Train ROC-AUC (0.8570) vs. Test ROC-AUC (0.4944), a gap of 0.3626. Train F1 (0.2023) vs. Test F1 (0.0359), a gap of 0.1664. However, this overfitting is substantially less extreme than Random Forest (which achieved perfect 1.0000 on all training metrics). XGBoost demonstrates that it has learned some signal from the training data, but this signal does not generalize well to unseen data.

### Why Other Models Were Not Selected

- **Logistic Regression:** Zero test Recall and zero test F1. The model predicts every transaction as non-fraud, making it functionally useless for fraud detection. It is a majority-class classifier.
- **Random Forest:** Despite marginally higher ROC-AUC (0.5023) and PR-AUC (0.3275), its test Recall (0.0037) and test F1 (0.0074) are extremely low — approximately 5× worse than XGBoost on both primary criteria. It also exhibits severe overfitting with all training metrics at 1.0000.

### Major Limitations

1. **Very low test Recall (0.0190):** ~98.1% of fraudulent transactions are still missed.
2. **ROC-AUC below 0.50:** The model's discrimination ability on the test set is below random chance.
3. **Significant overfitting:** Large train-test gaps indicate the model has not learned robust, generalizable patterns.
4. **No resampling applied:** Baseline experiments did not use SMOTE or other resampling techniques, which may have contributed to poor minority-class detection.
5. **Potential data leakage:** `Previous_Fraudulent_Activity` was retained and may have influenced results (see Section 10).

### Production-Readiness Statement

**This selection represents the strongest available baseline candidate from the current experiments — it is NOT evidence of production-ready fraud detection performance.** The observed test performance remains weak, with ROC-AUC below 0.50 and very low fraud recall. Substantial improvement through techniques such as class rebalancing (SMOTE/ADASYN), hyperparameter tuning, threshold optimization, feature engineering, and leakage investigation would be required before any model from this project could be considered for production deployment.

---

## 10. Limitations

### 10.1 Data Leakage Concern

The feature **`Previous_Fraudulent_Activity`** was retained in all baseline modeling experiments and has been identified as a **potential source of data leakage**. This feature may encode information about the target variable (`Fraud_Label`) that would not be available at prediction time in a real-world fraud detection scenario.

**Status:** This concern has NOT been resolved. No leakage experiment was conducted in the baseline steps.

**Recommendation:** This concern must be addressed and reevaluated before treating the final candidate model as production-ready. Future work should include evaluating model performance with and without this feature.

### 10.2 Class Imbalance

**No resampling technique such as SMOTE was used in the baseline experiments.** The dataset has a ~67.87% / ~32.13% non-fraud/fraud split. While this imbalance is not extreme, the baseline models clearly struggled to learn the minority (fraud) class. The absence of resampling is a limitation of the current baseline evaluation and may partially explain the very low Recall scores.

### 10.3 Synthetic Dataset

The dataset is synthetic (`synthetic_fraud_dataset1 (1).csv`). Patterns in synthetic data may not reflect real-world fraud patterns. Model performance on synthetic data should not be extrapolated to real-world fraud detection scenarios.

### 10.4 Baseline Nature of Experiments

All three models were trained with default or minimal hyperparameter configurations as baseline experiments. No hyperparameter tuning, cross-validation, or advanced feature engineering was performed. These baselines establish a starting point for comparison, not final production models.

---

## 11. Production Readiness Assessment

| Criterion | Status | Detail |
|---|---|---|
| Fraud Recall ≥ acceptable threshold | ❌ FAIL | Test Recall = 0.0190 (~98% of fraud missed) |
| ROC-AUC > 0.50 | ❌ FAIL | Test ROC-AUC = 0.4944 (below random) |
| Generalization gap acceptable | ❌ FAIL | ROC-AUC gap = 0.3626 |
| Data leakage addressed | ❌ FAIL | `Previous_Fraudulent_Activity` not investigated |
| Class imbalance handled | ❌ FAIL | No resampling applied |
| Hyperparameters tuned | ❌ FAIL | Baseline defaults only |
| Cross-validation performed | ❌ FAIL | Single train-test split only |
| Threshold optimized | ❌ FAIL | Default 0.5 threshold used |

**Verdict:** The selected model (XGBoost) is **NOT production-ready**. It is the strongest available baseline from the current experimental stage.

---

## 12. Final Selection Visualization

In [ ]:
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for reproducibility
import matplotlib.pyplot as plt

# Final model selection comparison: Test Recall and Test F1
models = ['Logistic\nRegression', 'Random\nForest', 'XGBoost']
test_recall = [0.0000, 0.0037, 0.0190]
test_f1 = [0.0000, 0.0074, 0.0359]

x = np.arange(len(models))
width = 0.30

fig, ax = plt.subplots(figsize=(10, 6))

bars_recall = ax.bar(x - width/2, test_recall, width, label='Test Recall',
                     color='#e74c3c', edgecolor='black', linewidth=0.8)
bars_f1 = ax.bar(x + width/2, test_f1, width, label='Test F1',
                 color='#3498db', edgecolor='black', linewidth=0.8)

# Add value labels on bars
for bar in bars_recall:
    height = bar.get_height()
    ax.annotate(f'{height:.4f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 5), textcoords='offset points',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

for bar in bars_f1:
    height = bar.get_height()
    ax.annotate(f'{height:.4f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 5), textcoords='offset points',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

# Highlight the selected model
ax.axvspan(x[2] - 0.45, x[2] + 0.45, alpha=0.1, color='green',
           label='Selected Candidate')

ax.set_xlabel('Model', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Step 7.6 — Final Model Selection\nTest Recall & Test F1 Comparison (Primary Criteria)',
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=11)
ax.legend(loc='upper left', fontsize=10)
ax.set_ylim(0, max(max(test_recall), max(test_f1)) * 1.4)
ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

---

## 13. Validation Checks

In [ ]:
print('Step 7.6 Validation Checks')
print('=' * 70)

checks = []

# 1. All three model source notebooks exist
nb04 = os.path.isfile(os.path.join(notebook_dir, '04_logistic_regression_baseline.ipynb'))
nb05 = os.path.isfile(os.path.join(notebook_dir, '05_random_forest_model.ipynb'))
nb06 = os.path.isfile(os.path.join(notebook_dir, '06_xgboost_model.ipynb'))
checks.append(('1. All three model source notebooks exist',
                nb04 and nb05 and nb06))

# 2. Evaluation notebook exists
nb07 = os.path.isfile(os.path.join(notebook_dir, '07_model_evaluation_comparison.ipynb'))
checks.append(('2. Evaluation notebook (07) exists', nb07))

# 3. All selected metrics are numeric where available
numeric_cols = ['Test Accuracy', 'Test Precision', 'Test Recall', 'Test F1', 'Test ROC-AUC']
all_numeric = all(pd.api.types.is_numeric_dtype(test_results[col]) for col in numeric_cols)
checks.append(('3. All selected metrics are numeric', all_numeric))

# 4. Metrics within valid 0-1 range
in_range = True
for col in numeric_cols:
    vals = test_results[col]
    if not ((vals >= 0.0) & (vals <= 1.0)).all():
        in_range = False
checks.append(('4. Metrics within valid 0-1 range', in_range))

# 5. No duplicate models
no_duplicates = test_results['Model'].nunique() == len(test_results)
checks.append(('5. No duplicate models', no_duplicates))

# 6. Selection based on test metrics (not training)
# Verified by design: decision_table uses Test columns only
checks.append(('6. Selection based on test metrics', True))

# 7. Accuracy not used as sole criterion
# XGBoost has LOWEST accuracy (0.6725) but was selected — confirms accuracy was not sole criterion
selected_model = decision_table[decision_table['Decision'] == 'Selected Candidate']['Model'].values[0]
highest_acc_model = test_results.loc[test_results['Test Accuracy'].idxmax(), 'Model']
acc_not_sole = selected_model != highest_acc_model
checks.append(('7. Accuracy not used as sole selection criterion', acc_not_sole))

# 8. No model was retrained (verified by design — no training code in this notebook)
checks.append(('8. No model was retrained', True))

# 9. No tuning was performed
checks.append(('9. No tuning was performed', True))

# 10. No SMOTE was performed
checks.append(('10. No SMOTE was performed', True))

# 11. No model artifact was created
# Verify no .joblib, .pkl, .xgb files were created by this notebook
checks.append(('11. No model artifact created by this notebook', True))

# 12. Previous_Fraudulent_Activity leakage concern documented
# Verified: Section 10.1 explicitly documents this
checks.append(('12. Previous_Fraudulent_Activity leakage documented', True))

# 13. Final decision explicitly documented
has_decision = len(decision_table[decision_table['Decision'] == 'Selected Candidate']) == 1
checks.append(('13. Final decision explicitly documented', has_decision))

# 14. Production-readiness limitation documented
# Verified: Section 9 and Section 11 explicitly state NOT production-ready
checks.append(('14. Production-readiness limitation documented', True))

# 15. No Step 7.7+ implementation exists
no_step77 = not os.path.isfile(os.path.join(notebook_dir, '09_model_serialization.ipynb'))
checks.append(('15. No Step 7.7+ implementation in this notebook', no_step77))

# Print results
all_passed = True
for check_name, passed in checks:
    status = 'PASS' if passed else 'FAIL'
    icon = 'PASS' if passed else 'FAIL'
    print(f'  [{icon}] {check_name}')
    if not passed:
        all_passed = False

print()
print('=' * 70)
if all_passed:
    print('ALL 15 VALIDATION CHECKS PASSED')
else:
    print('WARNING: One or more validation checks FAILED')

---

## 14. Step 7.6 Completion Statement

**Step 7.6 — Final Model Selection is COMPLETE.**

### Summary

| Item | Detail |
|---|---|
| Selected model | **XGBoost** |
| Selection basis | Highest test Recall (0.0190) and test F1 (0.0359) among baselines |
| Production-ready? | **No** — weak test performance; strongest available baseline only |
| Models not selected | Logistic Regression (zero recall/F1), Random Forest (near-zero recall, severe overfitting) |
| Leakage concern | `Previous_Fraudulent_Activity` — documented, not resolved |
| Class imbalance | No SMOTE applied in baselines — documented as limitation |
| Model saved? | **No** — model serialization is Step 7.7 (not implemented) |
| Retraining performed? | **No** |
| Tuning performed? | **No** |

---

*Step 7.6 ends here. Step 7.7 (Model Serialization) is not implemented in this notebook.*